# Realistic Synthetic VI Science Cycle

This notebook visualizes a full synthetic-to-inference science cycle output produced by `scripts/run_realistic_synthetic_vi_cycle.py`.

Run once in terminal first:

```bash
python scripts/run_realistic_synthetic_vi_cycle.py --output-dir outputs/realistic_synthetic_vi_cycle
```


In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

plt.style.use('default')
plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['axes.grid'] = True


In [ ]:
from pathlib import Path

candidate_dirs = [
    Path('../outputs'),
    Path('../outputs/realistic_synthetic_vi_cycle_consistent_sfh_ceh'),
]

output_dir = None
for d in candidate_dirs:
    if (d / 'summary.json').exists() and (d / 'science_cycle_outputs.npz').exists():
        output_dir = d
        break

if output_dir is None:
    raise FileNotFoundError('Could not find summary.json + science_cycle_outputs.npz in expected output directories.')

summary = json.loads((output_dir / 'summary.json').read_text())
arr = np.load(output_dir / 'science_cycle_outputs.npz')

print('Output dir:', output_dir)
print('Sampler:', summary.get('sampler'))
print('Forward model:', summary.get('config', {}).get('forward_model', 'unknown'))
print('Pipeline:', summary.get('config', {}).get('pipeline_name', 'unknown'))
print('VI converged:', summary.get('vi', {}).get('converged'))
print('Final objective:', summary.get('vi', {}).get('final_objective'))
print('Gradient rel-L2 error:', summary.get('gradient_check', {}).get('relative_l2_error'))

## 1. IC and Ground Truth Diagnostics


In [ ]:
coords = arr['coords_xy']
spaxels = arr['spaxels']
true_age = arr['true_age']
true_met = arr['true_metallicity']
mass = arr['mass'] if 'mass' in arr.files else np.ones_like(true_age)
velocity_xyz = arr['velocity_xyz'] if 'velocity_xyz' in arr.files else None
true_vz = arr['true_vz'] if 'true_vz' in arr.files else None
fit_vz = arr['fit_vz'] if 'fit_vz' in arr.files else None

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
sc0 = axs[0].scatter(coords[:, 0], coords[:, 1], c=true_age, s=20)
axs[0].set_title('Particle ICs colored by true age')
plt.colorbar(sc0, ax=axs[0], fraction=0.046)

sc1 = axs[1].scatter(coords[:, 0], coords[:, 1], c=true_met, s=20)
axs[1].set_title('Particle ICs colored by true metallicity')
plt.colorbar(sc1, ax=axs[1], fraction=0.046)

axs[2].hist2d(spaxels[:, 0], spaxels[:, 1], bins=24)
axs[2].set_title('Particle occupancy in spaxel grid')
axs[2].set_xlabel('ix')
axs[2].set_ylabel('iy')

plt.tight_layout()
plt.show()

## 2. Cube-Level Fit Diagnostics


In [ ]:
target = arr['target_cube']
pred = arr['pred_mean_cube']
resid = arr['residual_cube']
chi2 = arr['chi2_cube']
wave_idx = target.shape[2] // 2

fig, axs = plt.subplots(1, 4, figsize=(18, 4))
im0 = axs[0].imshow(target[:, :, wave_idx])
axs[0].set_title(f'Ground truth slice (w={wave_idx})')
plt.colorbar(im0, ax=axs[0], fraction=0.046)

im1 = axs[1].imshow(pred[:, :, wave_idx])
axs[1].set_title('Posterior mean slice')
plt.colorbar(im1, ax=axs[1], fraction=0.046)

im2 = axs[2].imshow(resid[:, :, wave_idx])
axs[2].set_title('Residual slice')
plt.colorbar(im2, ax=axs[2], fraction=0.046)

im3 = axs[3].imshow(chi2[:, :, wave_idx])
axs[3].set_title('Chi2 slice')
plt.colorbar(im3, ax=axs[3], fraction=0.046)

plt.tight_layout()
plt.show()


## 3. Variational Optimization Diagnostics


In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(18, 4))
axs[0].plot(arr['vi_objective'])
axs[0].set_title('VI objective')

axs[1].plot(arr['vi_reconstruction'])
axs[1].set_title('Reconstruction')

axs[2].plot(arr['vi_kl'])
axs[2].set_title('KL')

axs[3].plot(arr['vi_grad_norm'])
axs[3].set_yscale('log')
axs[3].set_title('Grad norm')

for ax in axs:
    ax.set_xlabel('step')
plt.tight_layout()
plt.show()


## 4. Truth-Recovery Checks


In [ ]:
fit_age = arr['fit_age']
fit_met = arr['fit_metallicity']

fig, axs = plt.subplots(1, 2, figsize=(11, 4))
axs[0].scatter(true_age, fit_age, s=16)
lims = [min(true_age.min(), fit_age.min()), max(true_age.max(), fit_age.max())]
axs[0].plot(lims, lims, 'k--')
axs[0].set_title('Age recovery')
axs[0].set_xlabel('truth')
axs[0].set_ylabel('fit')

axs[1].scatter(true_met, fit_met, s=16)
lims = [min(true_met.min(), fit_met.min()), max(true_met.max(), fit_met.max())]
axs[1].plot(lims, lims, 'k--')
axs[1].set_title('Metallicity recovery')
axs[1].set_xlabel('truth')
axs[1].set_ylabel('fit')

plt.tight_layout()
plt.show()

print('Recovery metrics:')
for k, v in summary['recovery'].items():
    print(f'  {k}: {v:.6g}')


## SFH+CEH Diagnostics

These diagnostics compare true vs recovered age-metallicity structure and against the configured CEH prior curve.


In [ ]:
cfg = summary['config']
age_min = cfg.get('synthetic_age_min_gyr', 0.5)
age_max = cfg.get('synthetic_age_max_gyr', 12.0)
z_old = cfg.get('synthetic_ceh_z_old', cfg.get('prior_ceh_z_old', 8e-4))
z_young = cfg.get('synthetic_ceh_z_young', cfg.get('prior_ceh_z_young', 8e-3))
gamma = cfg.get('synthetic_ceh_gamma', cfg.get('prior_ceh_gamma', 1.2))

def ceh_curve(age):
    frac = np.clip((age - age_min) / max(age_max - age_min, 1e-8), 0.0, 1.0)
    return z_young - (z_young - z_old) * (frac ** gamma)

def corrcoef_safe(x, y):
    x = np.asarray(x) - np.mean(x)
    y = np.asarray(y) - np.mean(y)
    denom = (np.sqrt(np.mean(x * x)) * np.sqrt(np.mean(y * y))) + 1e-12
    return float(np.mean(x * y) / denom)

true_corr = corrcoef_safe(true_age, true_met)
fit_corr = corrcoef_safe(fit_age, fit_met)
print('age-met corr (truth):', f'{true_corr:.3f}')
print('age-met corr (fit):  ', f'{fit_corr:.3f}')


In [ ]:
age_grid = np.linspace(age_min, age_max, 200)
z_grid = ceh_curve(age_grid)

true_ceh_resid = true_met - ceh_curve(true_age)
fit_ceh_resid = fit_met - ceh_curve(fit_age)

fig, ax = plt.subplots(1, 3, figsize=(18, 4))

ax[0].scatter(true_age, true_met, s=20, alpha=0.7, label='Truth')
ax[0].scatter(fit_age, fit_met, s=20, alpha=0.7, label='Recovered')
ax[0].plot(age_grid, z_grid, 'k--', lw=2, label='Target CEH prior')
ax[0].set_xlabel('Age [Gyr]')
ax[0].set_ylabel('Metallicity')
ax[0].set_title('Age-Metallicity Relation')
ax[0].legend()

ax[1].hist(true_ceh_resid, bins=16, alpha=0.7, label='Truth residual')
ax[1].hist(fit_ceh_resid, bins=16, alpha=0.7, label='Recovered residual')
ax[1].axvline(0.0, color='k', ls='--', lw=1.5)
ax[1].set_xlabel('Metallicity - CEH(age)')
ax[1].set_title('Residual Distribution to CEH')
ax[1].legend()

true_rmse = float(np.sqrt(np.mean(true_ceh_resid**2)))
fit_rmse = float(np.sqrt(np.mean(fit_ceh_resid**2)))
ax[2].bar(['Truth', 'Recovered'], [true_rmse, fit_rmse])
ax[2].set_ylabel('RMSE to CEH curve')
ax[2].set_title('CEH Consistency')

plt.tight_layout()
print('RMSE(truth to CEH):    ', f'{true_rmse:.6f}')
print('RMSE(recovered to CEH):', f'{fit_rmse:.6f}')


## Spatial + Kinematic Diagnostics

These are truth-space diagnostics for structure and kinematics. Current VI setup does **not** infer positions/velocities yet; it only infers age and metallicity with fixed particle phase-space coordinates.


In [ ]:
nx, ny, _ = target.shape
mass_map = np.zeros((nx, ny), dtype=float)
age_true_map = np.zeros((nx, ny), dtype=float)
age_fit_map = np.zeros((nx, ny), dtype=float)
met_true_map = np.zeros((nx, ny), dtype=float)
met_fit_map = np.zeros((nx, ny), dtype=float)
vz_true_map = np.zeros((nx, ny), dtype=float) if true_vz is not None else None
vz_fit_map = np.zeros((nx, ny), dtype=float) if fit_vz is not None else None
w_sum = np.zeros((nx, ny), dtype=float)

fit_age = arr['fit_age']
fit_met = arr['fit_metallicity']

for i in range(spaxels.shape[0]):
    ix, iy = int(spaxels[i, 0]), int(spaxels[i, 1])
    m = float(mass[i])
    mass_map[ix, iy] += m
    age_true_map[ix, iy] += m * float(true_age[i])
    age_fit_map[ix, iy] += m * float(fit_age[i])
    met_true_map[ix, iy] += m * float(true_met[i])
    met_fit_map[ix, iy] += m * float(fit_met[i])
    if vz_true_map is not None:
        vz_true_map[ix, iy] += m * float(true_vz[i])
    if vz_fit_map is not None:
        vz_fit_map[ix, iy] += m * float(fit_vz[i])
    w_sum[ix, iy] += m

mask = w_sum > 0
for mp in [age_true_map, age_fit_map, met_true_map, met_fit_map]:
    mp[mask] = mp[mask] / w_sum[mask]
if vz_true_map is not None:
    vz_true_map[mask] = vz_true_map[mask] / w_sum[mask]
if vz_fit_map is not None:
    vz_fit_map[mask] = vz_fit_map[mask] / w_sum[mask]

fig, ax = plt.subplots(2, 4, figsize=(18, 9))
im = ax[0, 0].imshow(mass_map.T, origin='lower', cmap='magma')
ax[0, 0].set_title('Mass map')
plt.colorbar(im, ax=ax[0, 0], fraction=0.046)

im = ax[0, 1].imshow(age_true_map.T, origin='lower', cmap='viridis')
ax[0, 1].set_title('Truth age map')
plt.colorbar(im, ax=ax[0, 1], fraction=0.046)

im = ax[0, 2].imshow(age_fit_map.T, origin='lower', cmap='viridis')
ax[0, 2].set_title('Recovered age map')
plt.colorbar(im, ax=ax[0, 2], fraction=0.046)

im = ax[0, 3].imshow((age_fit_map - age_true_map).T, origin='lower', cmap='bwr')
ax[0, 3].set_title('Age residual (fit-truth)')
plt.colorbar(im, ax=ax[0, 3], fraction=0.046)

im = ax[1, 0].imshow(met_true_map.T, origin='lower', cmap='cividis')
ax[1, 0].set_title('Truth metallicity map')
plt.colorbar(im, ax=ax[1, 0], fraction=0.046)

im = ax[1, 1].imshow(met_fit_map.T, origin='lower', cmap='cividis')
ax[1, 1].set_title('Recovered metallicity map')
plt.colorbar(im, ax=ax[1, 1], fraction=0.046)

im = ax[1, 2].imshow((met_fit_map - met_true_map).T, origin='lower', cmap='bwr')
ax[1, 2].set_title('Metallicity residual (fit-truth)')
plt.colorbar(im, ax=ax[1, 2], fraction=0.046)

if vz_true_map is not None and vz_fit_map is not None:
    im = ax[1, 3].imshow((vz_fit_map - vz_true_map).T, origin='lower', cmap='coolwarm')
    ax[1, 3].set_title('v_z residual map [km/s]')
    plt.colorbar(im, ax=ax[1, 3], fraction=0.046)
else:
    ax[1, 3].axis('off')

for a in ax.ravel():
    a.set_xlabel('spaxel x')
    a.set_ylabel('spaxel y')

plt.tight_layout()
plt.show()

In [ ]:
print('Above: spatial maps for mass, age, metallicity, and (if present) LOS velocity residuals.')
print('Below: particle-level kinematic recovery diagnostics.')

## Kinematic Recovery Diagnostics

Compare recovered line-of-sight velocity against truth (if velocity outputs are available).


In [ ]:
if (true_vz is None) or (fit_vz is None):
    print('No velocity recovery arrays in this output file yet. Re-run script after the v_z inference update.')
else:
    nx, ny, _ = target.shape
    vz_true_map = np.zeros((nx, ny), dtype=float)
    vz_fit_map = np.zeros((nx, ny), dtype=float)
    w_sum = np.zeros((nx, ny), dtype=float)

    for i in range(spaxels.shape[0]):
        ix, iy = int(spaxels[i, 0]), int(spaxels[i, 1])
        m = float(mass[i])
        vz_true_map[ix, iy] += m * float(true_vz[i])
        vz_fit_map[ix, iy] += m * float(fit_vz[i])
        w_sum[ix, iy] += m

    mask = w_sum > 0
    vz_true_map[mask] /= w_sum[mask]
    vz_fit_map[mask] /= w_sum[mask]

    vz_mae = float(np.mean(np.abs(fit_vz - true_vz)))
    vz_rmse = float(np.sqrt(np.mean((fit_vz - true_vz) ** 2)))
    corr = np.corrcoef(true_vz, fit_vz)[0, 1] if np.std(true_vz) > 0 and np.std(fit_vz) > 0 else np.nan

    fig, ax = plt.subplots(1, 4, figsize=(18, 4))
    im0 = ax[0].imshow(vz_true_map.T, origin='lower', cmap='coolwarm')
    ax[0].set_title('Truth v_z map')
    plt.colorbar(im0, ax=ax[0], fraction=0.046)

    im1 = ax[1].imshow(vz_fit_map.T, origin='lower', cmap='coolwarm')
    ax[1].set_title('Recovered v_z map')
    plt.colorbar(im1, ax=ax[1], fraction=0.046)

    im2 = ax[2].imshow((vz_fit_map - vz_true_map).T, origin='lower', cmap='bwr')
    ax[2].set_title('v_z residual map (fit-truth)')
    plt.colorbar(im2, ax=ax[2], fraction=0.046)

    ax[3].scatter(true_vz, fit_vz, s=18, alpha=0.8)
    lo = min(float(np.min(true_vz)), float(np.min(fit_vz)))
    hi = max(float(np.max(true_vz)), float(np.max(fit_vz)))
    ax[3].plot([lo, hi], [lo, hi], 'k--', lw=1.5)
    ax[3].set_xlabel('Truth v_z [km/s]')
    ax[3].set_ylabel('Recovered v_z [km/s]')
    ax[3].set_title('Particle-level v_z recovery')

    for a in ax[:3]:
        a.set_xlabel('spaxel x')
        a.set_ylabel('spaxel y')

    plt.tight_layout()
    plt.show()

    print(f'v_z MAE [km/s]:  {vz_mae:.6f}')
    print(f'v_z RMSE [km/s]: {vz_rmse:.6f}')
    print(f'v_z corr:        {corr:.6f}')